# Contextual Chunk Headers [Step 4 - Fix the Embedding, Not the Return]

> **MLCourse - Agentic AI - Advanced RAG - Contextual Retrieval**

Notebooks 02 and 03 fixed the small-to-big problem on the **return** side: index
something small, hand back something bigger. This notebook fixes it on the
**index** side.

The observation is simple. A chunk like

> "She said it was the stupidest tea-party she had ever been at in all her life."

is unretrievable for the query *"what did Alice think of the Mad Hatter's tea
party?"* - it contains neither "Alice" nor "Hatter". The information needed to
retrieve it lives in the surrounding document, and the embedding never saw it.

**Contextual chunk headers** fix that by prepending a short piece of situating
context to each chunk *before embedding*:

```
   Document: Alice's Adventures in Wonderland | Scene: the Mad Tea-Party.
   Alice, the Hatter, the March Hare and the Dormouse are at the table.
   ---
   She said it was the stupidest tea-party she had ever been at in all her life.
```

Anthropic published a variant of this in 2024 under the name **Contextual
Retrieval**, generating one header per chunk with an LLM and reporting large
reductions in retrieval failure. The idea works with cheap static headers too,
and this notebook builds both.

### 1. Setup


In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

# "Parent" units: paragraphs. Big enough to answer from, too big to retrieve
# precisely. These are the documents we will later cut into small children.
parents = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]

print("parent paragraphs:", len(parents))
print("mean parent length:", int(sum(len(p) for p in parents) / len(parents)), "chars")

parent paragraphs: 237
mean parent length: 379 chars


In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def build_index(texts):
    """Embed a list of texts and return the normalised matrix."""
    return encoder.encode(texts, normalize_embeddings=True,
                          batch_size=64, show_progress_bar=False)


def search(index, texts, query, top_n=5):
    """Return [(position, score)] of the best matches in `index`."""
    q = encoder.encode([query], normalize_embeddings=True)[0]
    sims = index @ q
    order = np.argsort(sims)[::-1][:top_n]
    return [(int(i), float(sims[i])) for i in order]


print("encoder ready:", encoder.get_sentence_embedding_dimension(), "dimensions")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

encoder ready: 384 dimensions


### 2. The problem, demonstrated

Let us find real chunks in our corpus that are unretrievable because their
subject was cut away.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=0)

chunks, chunk_parent = [], []
for parent_id, parent in enumerate(parents):
    for piece in splitter.split_text(parent):
        chunks.append(piece)
        chunk_parent.append(parent_id)

plain_vectors = build_index(chunks)
print(f"{len(chunks)} chunks of ~250 chars")

# Chunks that mention neither a character name nor a place: contextless orphans.
NAMES = re.compile(r"alice|rabbit|queen|hatter|caterpillar|cat|duchess|dormouse|"
                   r"gryphon|turtle|king|knave", re.IGNORECASE)
orphans = [i for i, c in enumerate(chunks) if not NAMES.search(c)]
print(f"chunks with no character or place mentioned: {len(orphans)}\n")
for i in orphans[:3]:
    print(f"  chunk {i}: {chunks[i][:130]}...")

473 chunks of ~250 chars
chunks with no character or place mentioned: 188

  chunk 10: Either the well was very deep, or she fell very slowly, for she had plenty of time as she went down to look about her and to wonde...
  chunk 11: to see anything; then she looked at the sides of the well, and noticed that they were filled with cupboards and book-shelves; here...
  chunk 12: labelled “ORANGE MARMALADE”, but to her great disappointment it was empty: she did not like to drop the jar for fear of killing so...


Those chunks may contain the answer to a question, but no query about a
character will ever find them. They are dead weight in the index.

### 3. Static headers: cheap and surprisingly effective

The cheapest header uses metadata you already have - title, chapter, section
path. No LLM call, no cost, computed at index time from document structure. In a
real corpus this is your heading hierarchy, your file path, your ticket title.

We derive a crude chapter label from the corpus and attach the parent's opening
words as a locator.

In [6]:
CHAPTER_RE = re.compile(r"^CHAPTER\s+([IVXL]+)\.?\s*(.*)$", re.IGNORECASE)

chapter_of_parent, current = {}, "Front matter"
for parent_id, parent in enumerate(parents):
    m = CHAPTER_RE.match(parent.strip())
    if m:
        current = f"Chapter {m.group(1)} - {m.group(2).strip()[:40]}"
    chapter_of_parent[parent_id] = current

print("chapters detected:", len(set(chapter_of_parent.values())))
for name in list(dict.fromkeys(chapter_of_parent.values()))[:5]:
    print("  -", name)


def static_header(chunk_id):
    pid = chunk_parent[chunk_id]
    return (f"Document: Alice's Adventures in Wonderland. "
            f"{chapter_of_parent[pid]}. "
            f"Passage begins: {parents[pid][:60]}...")


static_texts = [static_header(i) + "\n---\n" + c for i, c in enumerate(chunks)]
static_vectors = build_index(static_texts)

print("\nexample of a headed chunk:\n")
print(static_texts[orphans[0]][:330], "...")

chapters detected: 1
  - Chapter I - Down the Rabbit-Hole CHAPTER II. The Poo



example of a headed chunk:

Document: Alice's Adventures in Wonderland. Chapter I - Down the Rabbit-Hole CHAPTER II. The Poo. Passage begins: Either the well was very deep, or she fell very slowly, for ...
---
Either the well was very deep, or she fell very slowly, for she had plenty of time as she went down to look about her and to wonder what was going t ...


### 4. LLM-generated headers: expensive and better

A static header says *where* the chunk lives. An LLM header can say *what it is
about* - resolving pronouns, naming the participants, summarising the situation.
That is much more useful, and much more expensive: one LLM call per chunk at
index time.

The economics: indexing is a **one-off** cost amortised over every future query,
so a few thousand calls is often perfectly acceptable. Indexing a million chunks
is not. Anthropic's version keeps this affordable with prompt caching of the
document body across all its chunks.

We generate headers for a small sample here, pacing calls against the free-tier
budget.

In [7]:
CONTEXT_PROMPT = (
    "Here is a passage from Lewis Carroll's 'Alice's Adventures in Wonderland':\n"
    "<document>\n{parent}\n</document>\n\n"
    "Here is a chunk taken from that passage:\n<chunk>\n{chunk}\n</chunk>\n\n"
    "Write ONE short sentence (max 25 words) that situates this chunk within the "
    "passage: name who is involved and what is happening, resolving any pronouns. "
    "Output only that sentence."
)

SAMPLE = orphans[:6] + [i for i in range(len(chunks)) if i not in orphans][:6]
SAMPLE = sorted(set(SAMPLE))

llm_headers = {}
for n, cid in enumerate(SAMPLE, 1):
    llm_headers[cid] = ask(CONTEXT_PROMPT.format(
        parent=parents[chunk_parent[cid]][:1200], chunk=chunks[cid]))
    print(f"[{n}/{len(SAMPLE)}] chunk {cid}: {llm_headers[cid]}")
    time.sleep(2.0)          # pacing: Groq free tier is ~8000 tokens/minute

[1/12] chunk 0: This chunk lists the first seven chapter titles of Lewis Carroll's novel, Alice's Adventures in Wonderland.


[2/12] chunk 1: These chapters conclude Alice's adventures in Wonderland, detailing her encounters with the Queen, the Mock Turtle, and the final trial.


[3/12] chunk 2: Alice, bored on the riverbank, questions the value of her sister's book because it lacks pictures or conversations.


[4/12] chunk 3: Alice thinks a book is useless if it lacks pictures or conversations.


[5/12] chunk 4: Alice is considering making a daisy-chain when a White Rabbit suddenly runs past her.


...

[8/12] chunk 11: Alice looks at the well's sides, sees shelves, and takes down a jar as she falls.


[9/12] chunk 12: Alice, falling down the well, places an empty orange marmalade jar into a cupboard to avoid injuring anyone below.


[10/12] chunk 14: The narrator confirms that Alice's claim about not mentioning a fall from the house was likely true.


[11/12] chunk 15: Alice falls down a hole, wondering if the fall will end and estimating she is near the earth's center.


[12/12] chunk 18: Alice imagines falling through the earth to the Antipathies while wondering if she is in New Zealand or Australia.


Look at what those headers added: character names, the scene, the resolved
antecedent of every "she" and "it". That is exactly the retrieval signal the raw
chunk was missing.

### 5. Does it change retrieval? Measure on the sample.

We build a small three-way index over the sampled chunks - plain, static header,
LLM header - and compare how each ranks for questions aimed at the orphan chunks.

In [8]:
import numpy as np

sample_plain = [chunks[i] for i in SAMPLE]
sample_static = [static_header(i) + "\n---\n" + chunks[i] for i in SAMPLE]
sample_llm = [llm_headers[i] + "\n---\n" + chunks[i] for i in SAMPLE]

v_plain = build_index(sample_plain)
v_static = build_index(sample_static)
v_llm = build_index(sample_llm)

PROBES = [
    "What did Alice think of the Mad Hatter's tea party?",
    "What did the Queen of Hearts order to be done?",
    "What happened when Alice changed size?",
]

print(f"{'probe':<52}{'plain':>9}{'static':>9}{'llm':>9}")
print("-" * 80)
for probe in PROBES:
    q = encoder.encode([probe], normalize_embeddings=True)[0]
    print(f"{probe[:50]:<52}"
          f"{float((v_plain @ q).max()):>9.3f}"
          f"{float((v_static @ q).max()):>9.3f}"
          f"{float((v_llm @ q).max()):>9.3f}")

probe                                                   plain   static      llm
--------------------------------------------------------------------------------
What did Alice think of the Mad Hatter's tea party      0.518    0.564    0.536
What did the Queen of Hearts order to be done?          0.378    0.292    0.315
What happened when Alice changed size?                  0.472    0.458    0.493


In [9]:
# Which chunk each variant considers the best match - the ordering, not just the score.
probe = PROBES[0]
q = encoder.encode([probe], normalize_embeddings=True)[0]

print("query:", probe, "\n")
for label, vecs in [("plain", v_plain), ("static header", v_static),
                    ("llm header", v_llm)]:
    best = int(np.argmax(vecs @ q))
    print(f"{label:<15} -> chunk {SAMPLE[best]} (cos={float((vecs @ q)[best]):.3f})")
    print(f"{'':15}    {chunks[SAMPLE[best]][:110]}...")
    print()

query: What did Alice think of the Mad Hatter's tea party? 

plain           -> chunk 3 (cos=0.518)
                   book,” thought Alice “without pictures or conversations?”...

static header   -> chunk 4 (cos=0.564)
                   So she was considering in her own mind (as well as she could, for the hot day made her feel very sleepy and st...

llm header      -> chunk 4 (cos=0.536)
                   So she was considering in her own mind (as well as she could, for the hot day made her feel very sleepy and st...



### 6. Header in, header out?

An important design choice: the header helped **retrieval**, but should it be in
the text you send to the LLM?

- **Keep it** when it adds real grounding the chunk lacks ("this is from the
  2023 annual report, section 4"). It helps the model attribute and reason.
- **Drop it** when it is redundant with what you already put in the prompt, or
  when context budget is tight. Store the raw chunk alongside the headed one and
  return whichever you want.

The pattern is: embed the headed version, store both, return whichever suits.

In [10]:
def headed_rag(question, use_header_in_context=True, top_k=3):
    q = encoder.encode([question], normalize_embeddings=True)[0]
    order = np.argsort(v_llm @ q)[::-1][:top_k]
    pieces = []
    for pos in order:
        cid = SAMPLE[pos]
        pieces.append((llm_headers[cid] + "\n" + chunks[cid])
                      if use_header_in_context else chunks[cid])
    context = "\n\n".join(f"[{i}] {p}" for i, p in enumerate(pieces))
    return ask(
        "Answer using ONLY the context. If it is insufficient, say so.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )


q = "What did Alice think of the Mad Hatter's tea party?"
print("WITH headers in context:")
print(headed_rag(q, use_header_in_context=True))

WITH headers in context:


The provided context is insufficient to answer the question.


In [11]:
print("WITHOUT headers in context (same chunks):")
print(headed_rag(q, use_header_in_context=False))

WITHOUT headers in context (same chunks):


The provided context is insufficient to answer the question.


### 7. Pitfalls

- **Indexing cost is linear in chunks.** One LLM call per chunk. Budget it, use
  prompt caching where the provider supports it, and consider generating headers
  only for chunks that need them (the orphans).
- **Headers must be regenerated when documents change.** A stale header is worse
  than none - it asserts context that is no longer true.
- **Do not over-write the chunk.** The header situates; it must not paraphrase
  the chunk's content, or the embedding starts matching the summary rather than
  the text and you lose specificity.
- **Keep them short.** 15-25 words. A long header dominates the embedding of a
  short chunk - the tail wagging the dog.
- **Headers help BM25 too**, often more than they help dense search, because
  they inject the literal keywords (names, section titles) that keyword search
  needs. Worth remembering when you combine this with
  [`../01_hybrid_search`](../01_hybrid_search/README.md).

### 8. Key takeaways

- Contextual headers fix retrieval at the **embedding** stage rather than the
  return stage, so they compose with parent-document and sentence-window
  retrieval.
- **Static headers** (title, chapter, section path) are free and already help.
- **LLM-generated headers** resolve pronouns and name participants; they cost one
  call per chunk at index time and are usually the better version.
- Store the raw chunk too, and decide separately whether the header goes into
  the prompt.

Next: [`05_comparing_granularity.ipynb`](05_comparing_granularity.ipynb) puts all
four strategies on one evaluation set.